# Evaluate ASVspoof 2021 DF

This notebook updates ASVspoof 2021 DF `results.pkl`. Existing AASIST and LFCC+LCNN entries are reused unless forced; AASIST3 and AASIST-L are evaluated when missing.

## Dataset structure

The Kaggle ASVspoof 2021 package stores DF eval audio in `ASVspoof2021_DF_eval_part00/01/02` and metadata in `DF-keys-full/.../trial_metadata.txt`. The label is field index 5, not the last field. This notebook follows the same DF parsing used by the earlier AASIST and LFCC+LCNN runs.

In [ ]:
import gc
import importlib
import io
import json
import os
import pickle
import subprocess
import sys
import tarfile
import time
import base64
from pathlib import Path

import numpy as np
import pandas as pd

# A failed import can leave torch half-initialized in the notebook kernel.
# Clear that stale state before retrying imports in the same session.
_torch_mod = sys.modules.get("torch")
if _torch_mod is not None and not hasattr(_torch_mod, "Tensor"):
    for _name in list(sys.modules):
        if _name == "torch" or _name.startswith("torch."):
            del sys.modules[_name]

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from sklearn.metrics import roc_curve
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import soundfile as sf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "soundfile"])
    import soundfile as sf

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


def ensure_parent(path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def compute_eer(scores, labels) -> float:
    """Compute EER in percent. Higher score means more bonafide."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) * 50.0)


def load_pickle_results(candidates: list[Path]) -> dict:
    """Load the first existing results.pkl candidate."""
    for path in candidates:
        if path.exists():
            print(f"loading existing results: {path}")
            with open(path, "rb") as handle:
                return pickle.load(handle)
    return {}


def save_results_pickle(results: dict, output_path: Path) -> None:
    """Persist the unified dataset results file after each model finishes."""
    slim = {}
    for model_name, result in results.items():
        slim[model_name] = {
            "eer": float(result["eer"]),
            "scores": np.asarray(result["scores"], dtype=np.float64),
            "labels": np.asarray(result["labels"], dtype=np.int64),
        }
    ensure_parent(output_path)
    with open(output_path, "wb") as handle:
        pickle.dump(slim, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"saved {output_path} with models={list(slim)}")


def partial_path(output_dir: Path, model_name: str) -> Path:
    safe = model_name.replace("+", "_").replace(" ", "_").replace("/", "_")
    return output_dir / f"{safe}.partial.npz"


def load_partial(output_dir: Path, model_name: str, fallback_dirs: list[Path] | None = None) -> dict | None:
    path = partial_path(output_dir, model_name)
    if not path.exists():
        for fb_dir in (fallback_dirs or []):
            candidate = partial_path(fb_dir, model_name)
            if candidate.exists():
                path = candidate
                print(f"{model_name}: resuming partial from input dataset: {path}")
                break
    if not path.exists():
        return None
    data = np.load(path, allow_pickle=True)
    return {
        "scores": data["scores"].astype(np.float64).tolist(),
        "labels": data["labels"].astype(np.int64).tolist(),
        "utt_ids": data["utt_ids"].astype(str).tolist(),
    }


def save_partial(output_dir: Path, model_name: str, scores: list, labels: list, utt_ids: list) -> None:
    path = partial_path(output_dir, model_name)
    np.savez(
        ensure_parent(path),
        scores=np.asarray(scores, dtype=np.float64),
        labels=np.asarray(labels, dtype=np.int64),
        utt_ids=np.asarray(utt_ids, dtype=str),
    )


def clear_partial(output_dir: Path, model_name: str) -> None:
    path = partial_path(output_dir, model_name)
    if path.exists():
        path.unlink()


def release_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
DATASET_KEY = "asvspoof21"
OUTPUT_DIR = Path("/kaggle/working/asvspoof21")
OUTPUT_PKL = OUTPUT_DIR / "results.pkl"
INPUT_RESULTS = [
    OUTPUT_PKL,
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof21/results.pkl"),
    Path("/kaggle/input/sdd-survey/asvspoof21/results.pkl"),
    Path("results/asvspoof21/results.pkl"),
]
FORCE_EVAL = {
    "AASIST": False,
    "AASIST-L": False,
    "AASIST3": False,
    "LFCC+LCNN": False,
    "XLS-R+AASIST": False,
    "XLS-R+Nes2Net": False,
}
ENABLED_MODELS = ["AASIST", "LFCC+LCNN", "AASIST3", "AASIST-L", "XLS-R+AASIST", "XLS-R+Nes2Net"]
SMOKE_TEST_N = None
PARTIAL_SAVE_EVERY = 10000
NUM_WORKERS = 2
RUN_COMPOUND_SSL_MODELS = False
PARTIAL_INPUT_DIRS = [
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof21"),
    Path("/kaggle/input/sdd-survey/asvspoof21"),
    Path("/kaggle/input/datasets/minhbhm/sdd-partials-asv21"),
    Path("/kaggle/input/sdd-partials-asv21"),
]


def locate_asvspoof2021_df() -> dict:
    """Locate ASVspoof 2021 DF eval audio parts and trial metadata."""
    bases = [
        Path("/kaggle/input/datasets/mohammedabdeldayem/avsspoof-2021"),
        Path("/kaggle/input/avsspoof-2021"),
        Path("/kaggle/input/asvspoof-2021"),
    ]
    base = next((p for p in bases if p.exists()), None)
    if base is None:
        raise FileNotFoundError("ASVspoof 2021 root was not found. Attach mohammedabdeldayem/avsspoof-2021.")

    audio_dirs = []
    for part in ["ASVspoof2021_DF_eval_part00", "ASVspoof2021_DF_eval_part01", "ASVspoof2021_DF_eval_part02"]:
        root = base / part
        if not root.exists():
            continue
        for dirpath, _, files in os.walk(root):
            if any(name.endswith(".flac") for name in files):
                audio_dirs.append(Path(dirpath))
                break

    trial = base / "DF-keys-full" / "keys" / "DF" / "CM" / "trial_metadata.txt"
    if not trial.exists():
        for dirpath, _, files in os.walk(base / "DF-keys-full"):
            for name in files:
                if "trial" in name.lower() and name.endswith(".txt"):
                    trial = Path(dirpath) / name
                    break
            if trial.exists():
                break
    if not audio_dirs or not trial.exists():
        raise FileNotFoundError("Could not locate ASVspoof 2021 DF audio parts or trial metadata.")
    return {"base": base, "audio_dirs": audio_dirs, "trial": trial}


def parse_asvspoof2021_df(trial_path: Path, audio_dirs: list[Path]) -> pd.DataFrame:
    """Parse ASVspoof 2021 DF. The CM key is field index 5."""
    audio_index = {}
    for audio_dir in audio_dirs:
        for name in os.listdir(audio_dir):
            if name.endswith(".flac"):
                audio_index[name[:-5]] = str(audio_dir / name)

    rows = []
    with open(trial_path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            label_text = parts[5]
            if label_text not in ("bonafide", "spoof"):
                continue
            utt_id = parts[1]
            if utt_id not in audio_index:
                continue
            rows.append({
                "speaker": parts[0],
                "utt_id": utt_id,
                "codec": parts[2],
                "source": parts[3] if len(parts) > 3 else "unknown",
                "attack": parts[4] if len(parts) > 4 else "unknown",
                "vocoder": parts[8] if len(parts) > 8 else "unknown",
                "label": 1 if label_text == "bonafide" else 0,
                "audio_path": audio_index[utt_id],
            })
    return pd.DataFrame(rows).reset_index(drop=True)


loc = locate_asvspoof2021_df()
print(loc)
eval_df = parse_asvspoof2021_df(loc["trial"], loc["audio_dirs"])
if SMOKE_TEST_N is not None:
    eval_df = eval_df.iloc[:SMOKE_TEST_N].copy()
print(eval_df["label"].value_counts().rename({1: "bonafide", 0: "spoof"}))
print(f"ASVspoof 2021 DF eval rows with audio: {len(eval_df):,}")

results = load_pickle_results(INPUT_RESULTS)

In [ ]:
import gc
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np

_torch_mod = sys.modules.get("torch")
if _torch_mod is not None and not hasattr(_torch_mod, "Tensor"):
    for _name in list(sys.modules):
        if _name == "torch" or _name.startswith("torch."):
            del sys.modules[_name]

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset


class SimpleLCNN(nn.Module):
    """Small LFCC+LCNN baseline. This must match the saved checkpoint architecture."""

    def __init__(self, n_lfcc: int = 60):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


LFCC_TRANSFORM = torchaudio.transforms.LFCC(
    sample_rate=16000,
    n_lfcc=60,
    speckwargs={"n_fft": 512, "hop_length": 160, "win_length": 320},
)


def run_shell(command: str) -> None:
    """Notebook-friendly shell runner."""
    print(command)
    rc = os.system(command)
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {command}")


def ensure_aasist_repo(repo_dir: Path = Path("/kaggle/working/aasist")) -> Path:
    """Clone clovaai/aasist only when missing."""
    if not repo_dir.exists():
        run_shell(f"git clone https://github.com/clovaai/aasist.git {repo_dir}")
    repo_str = str(repo_dir)
    if repo_str in sys.path:
        sys.path.remove(repo_str)
    sys.path.insert(0, repo_str)
    return repo_dir


def ensure_aasist3_repo(repo_dir: Path = Path("/kaggle/working/AASIST3")) -> Path:
    """Clone mtuciru/AASIST3 only when missing."""
    if not repo_dir.exists():
        run_shell(f"git clone https://github.com/mtuciru/AASIST3.git {repo_dir}")
    repo_str = str(repo_dir)
    if repo_str in sys.path:
        sys.path.remove(repo_str)
    sys.path.insert(0, repo_str)
    return repo_dir


def load_aasist_model(config_name: str, weight_name: str) -> nn.Module:
    """Load AASIST or AASIST-L from the official repository."""
    repo_dir = ensure_aasist_repo()
    cwd = Path.cwd()
    try:
        os.chdir(repo_dir)
        _reset_module_namespace(["models"])
        with open(repo_dir / "config" / config_name, "r", encoding="utf-8") as handle:
            cfg = json.load(handle)
        module = importlib.import_module(f"models.{cfg['model_config']['architecture']}")
        model = module.Model(cfg["model_config"]).to(DEVICE)
        state = torch.load(repo_dir / "models" / "weights" / weight_name, map_location=DEVICE)
        model.load_state_dict(state)
        return model.eval()
    finally:
        os.chdir(cwd)


def load_aasist3_model() -> nn.Module:
    """Load AASIST3 from Hugging Face through the AASIST3 repository wrapper."""
    repo_dir = ensure_aasist3_repo()
    cwd = Path.cwd()
    try:
        os.chdir(repo_dir)
        _reset_module_namespace(["model"])
        from model import aasist3
        return aasist3.from_pretrained("MTUCI/AASIST3").to(DEVICE).eval()
    finally:
        os.chdir(cwd)


SSL_AASIST_REPO_URL = "https://github.com/TakHemlata/SSL_Anti-spoofing.git"
SSL_AASIST_REPO_DIR = Path("/kaggle/working/SSL_Anti-spoofing")
NES2NET_REPO_URL = "https://github.com/Liu-Tianchi/Nes2Net_ASVspoof_ITW.git"
NES2NET_REPO_DIR = Path("/kaggle/working/Nes2Net_ASVspoof_ITW")
XLSR_300M_URL = "https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr2_300m.pt"


def install_pkg(pip_name: str, import_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


def _swap_sys_path(repo_dir: Path) -> None:
    repo_str = str(repo_dir)
    if repo_str in sys.path:
        sys.path.remove(repo_str)
    sys.path.insert(0, repo_str)


def ensure_ssl_aasist_repo() -> Path:
    """Clone TakHemlata/SSL_Anti-spoofing only when missing."""
    if not SSL_AASIST_REPO_DIR.exists():
        run_shell(f"git clone {SSL_AASIST_REPO_URL} {SSL_AASIST_REPO_DIR}")
    _swap_sys_path(SSL_AASIST_REPO_DIR)
    return SSL_AASIST_REPO_DIR


def ensure_nes2net_repo() -> Path:
    """Clone Liu-Tianchi/Nes2Net_ASVspoof_ITW only when missing."""
    if not NES2NET_REPO_DIR.exists():
        run_shell(f"git clone {NES2NET_REPO_URL} {NES2NET_REPO_DIR}")
    _swap_sys_path(NES2NET_REPO_DIR)
    return NES2NET_REPO_DIR


def ensure_repo_fairseq(repo_dir: Path) -> None:
    """Install the repository-pinned fairseq when available.

    The SSL repos were written against a specific fairseq snapshot. PyPI fairseq
    can import but still be ABI/API-incompatible on Kaggle, so prefer the bundled
    editable install and use PyPI only as a last resort.
    """
    candidates = [p for p in repo_dir.iterdir() if p.is_dir() and p.name.startswith("fairseq")]
    if candidates:
        fairseq_dir = candidates[0]
        patch_fairseq_for_python312(fairseq_dir)
        print(f"installing repo-pinned fairseq from {fairseq_dir}")
        _swap_sys_path(fairseq_dir)
        # Kaggle currently ships a recent pip that rejects old omegaconf metadata
        # used by this fairseq snapshot. Downgrading pip and using the legacy
        # resolver keeps the install reproducible without changing the repo code.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pip<24.1"])
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--use-deprecated=legacy-resolver",
            "-e",
            str(fairseq_dir),
        ])
        patch_fairseq_for_python312(fairseq_dir)
        patch_hydra_for_python312()
        _reset_module_namespace(["fairseq", "hydra", "omegaconf"])
        importlib.invalidate_caches()
        try:
            import fairseq  # noqa: F401
            print(f"fairseq import path: {fairseq.__file__}")
        except ImportError as exc:
            raise ImportError(
                f"fairseq install finished but import still failed. "
                f"fairseq_dir={fairseq_dir}, sys.path[0]={sys.path[0]}"
            ) from exc
        return

    install_pkg("fairseq")
    patch_hydra_for_python312()
    _reset_module_namespace(["fairseq", "hydra", "omegaconf"])
    importlib.invalidate_caches()
    import fairseq  # noqa: F401
    print(f"fairseq import path: {fairseq.__file__}")


def patch_fairseq_for_python312(fairseq_dir: Path) -> None:
    """Patch old fairseq dataclasses so they import on Python 3.11/3.12.

    The pinned fairseq snapshot uses mutable dataclass defaults such as
    `common: CommonConfig = CommonConfig()`. Newer Python rejects that pattern.
    For this notebook we only rewrite config defaults to `field(default_factory=...)`.
    """
    configs_path = fairseq_dir / "fairseq" / "dataclass" / "configs.py"
    if not configs_path.exists():
        return

    n_replacements = patch_dataclass_mutable_defaults(configs_path)
    if n_replacements:
        print(f"patched {n_replacements} fairseq dataclass defaults for Python 3.12: {configs_path}")
    patch_fairseq_hydra_init_for_default_factory(fairseq_dir)
    patch_fairseq_numpy_aliases(fairseq_dir)


def patch_fairseq_hydra_init_for_default_factory(fairseq_dir: Path) -> None:
    """Make fairseq hydra_init understand dataclass default_factory fields.

    After Python 3.12 compatibility patching, fields such as `common` no longer
    have `.default`; they have `.default_factory`. Old fairseq hydra_init only
    reads `.default`, so it passes dataclasses.MISSING into OmegaConf. This patch
    instantiates default_factory values before registering them with Hydra.
    """
    init_path = fairseq_dir / "fairseq" / "dataclass" / "initialize.py"
    if not init_path.exists():
        return
    text = init_path.read_text(encoding="utf-8")
    if "patched_py312_default_factory" in text:
        return

    if "from dataclasses import MISSING" not in text:
        text = "from dataclasses import MISSING\n" + text

    old = "v = FairseqConfig.__dataclass_fields__[k].default"
    new = (
        "field_info = FairseqConfig.__dataclass_fields__[k]\n"
        "        if field_info.default is not MISSING:\n"
        "            v = field_info.default\n"
        "        elif field_info.default_factory is not MISSING:\n"
        "            v = field_info.default_factory()\n"
        "        else:\n"
        "            v = MISSING"
    )
    if old in text:
        text = text.replace(old, new)
        text += "\n# patched_py312_default_factory\n"
        init_path.write_text(text, encoding="utf-8")
        print(f"patched fairseq hydra_init default_factory handling: {init_path}")


def patch_hydra_for_python312() -> None:
    """Patch hydra 1.0.x dataclass defaults installed as a fairseq dependency."""
    spec = importlib.util.find_spec("hydra")
    if spec is None or not spec.submodule_search_locations:
        return
    hydra_root = Path(list(spec.submodule_search_locations)[0])
    conf_path = hydra_root / "conf" / "__init__.py"
    if not conf_path.exists():
        return
    n_replacements = patch_dataclass_mutable_defaults(conf_path)
    if n_replacements:
        print(f"patched {n_replacements} hydra dataclass defaults for Python 3.12: {conf_path}")


def patch_dataclass_mutable_defaults(path: Path) -> int:
    """Rewrite common mutable dataclass defaults for Python 3.11/3.12 compatibility."""
    import re

    text = path.read_text(encoding="utf-8")
    original = text

    if "from dataclasses import" in text and "field" not in text.split("from dataclasses import", 1)[1].split("\n", 1)[0]:
        text = re.sub(
            r"^(from dataclasses import )([^\n]+)$",
            lambda m: m.group(1) + m.group(2).rstrip() + ", field",
            text,
            count=1,
            flags=re.MULTILINE,
        )
    elif "from dataclasses import" not in text:
        text = "from dataclasses import field\n" + text

    n_total = 0
    # Pattern: name: SomeConfig = SomeConfig()
    obj_pattern = re.compile(
        r"^(?P<indent>\s*)(?P<name>\w+):\s+(?P<cls>[\w.]+)\s*=\s*(?P=cls)\(\)\s*$",
        flags=re.MULTILINE,
    )
    text, n_obj = obj_pattern.subn(
        lambda m: (
            f"{m.group('indent')}{m.group('name')}: {m.group('cls')} = "
            f"field(default_factory={m.group('cls')})"
        ),
        text,
    )
    n_total += n_obj

    # Pattern: name: List[T] = [] / name: Dict[K, V] = {}
    list_pattern = re.compile(r"^(?P<indent>\s*)(?P<name>\w+):\s+List\[(?P<t>[^\]]+)\]\s*=\s*\[\]\s*$", flags=re.MULTILINE)
    text, n_list = list_pattern.subn(
        lambda m: f"{m.group('indent')}{m.group('name')}: List[{m.group('t')}] = field(default_factory=list)",
        text,
    )
    n_total += n_list

    dict_pattern = re.compile(r"^(?P<indent>\s*)(?P<name>\w+):\s+Dict\[(?P<t>[^\]]+)\]\s*=\s*\{\}\s*$", flags=re.MULTILINE)
    text, n_dict = dict_pattern.subn(
        lambda m: f"{m.group('indent')}{m.group('name')}: Dict[{m.group('t')}] = field(default_factory=dict)",
        text,
    )
    n_total += n_dict

    if text != original:
        path.write_text(text, encoding="utf-8")
    return n_total


def patch_fairseq_numpy_aliases(fairseq_dir: Path) -> None:
    """Replace removed numpy scalar type aliases across the fairseq bundle.

    NumPy 1.24 removed np.float, np.int, np.bool, np.complex, np.object, np.str.
    The pinned fairseq snapshot still uses these in several files (e.g. indexed_dataset.py).
    """
    import re
    _aliases = [
        (re.compile(r'\bnp\.float\b'), 'np.float64'),
        (re.compile(r'\bnp\.int\b'), 'np.int_'),
        (re.compile(r'\bnp\.bool\b'), 'np.bool_'),
        (re.compile(r'\bnp\.complex\b'), 'np.complex128'),
        (re.compile(r'\bnp\.object\b'), 'object'),
        (re.compile(r'\bnp\.str\b'), 'np.str_'),
    ]
    total = 0
    for py_file in fairseq_dir.rglob("*.py"):
        try:
            text = py_file.read_text(encoding="utf-8")
        except Exception:
            continue
        original = text
        for pattern, replacement in _aliases:
            text = pattern.sub(replacement, text)
        if text != original:
            py_file.write_text(text, encoding="utf-8")
            total += 1
    if total:
        print(f"patched numpy deprecated aliases in {total} fairseq files")


def find_or_download_xlsr_300m() -> Path:
    """Locate or download the fairseq XLS-R 300M base SSL checkpoint."""
    candidates = [
        Path("/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/xlsr/xlsr2_300m.pt"),
        Path("/kaggle/input/datasets/minhbhm/sdd-survey/ssl_models/xlsr2_300m.pt"),
        Path("/kaggle/input/sdd-survey/checkpoints/xlsr/xlsr2_300m.pt"),
        Path("/kaggle/input/xlsr-300m/xlsr2_300m.pt"),
        Path("/kaggle/working/xlsr2_300m.pt"),
    ]
    for path in candidates:
        if path.exists():
            print(f"found XLS-R 300M base SSL: {path}")
            return path
    target = Path("/kaggle/working/xlsr2_300m.pt")
    print(f"downloading XLS-R 300M (~1.2 GB) to {target}")
    run_shell(f"wget -q -O {target} {XLSR_300M_URL}")
    return target


def _link_xlsr_into(repo_dir: Path) -> None:
    """Both SSL_Anti-spoofing and Nes2Net hard-code ./xlsr2_300m.pt — point it at the cached file."""
    target = repo_dir / "xlsr2_300m.pt"
    if target.exists():
        return
    src = find_or_download_xlsr_300m()
    try:
        target.symlink_to(src)
    except (OSError, NotImplementedError):
        import shutil
        shutil.copy(src, target)


def find_xlsr_aasist_checkpoint() -> Path:
    """Locate TakHemlata's pretrained Wav2Vec2-XLSR+AASIST anti-spoofing weights."""
    bases = [
        Path("/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/xlsr_aasist"),
        Path("/kaggle/input/sdd-survey/checkpoints/xlsr_aasist"),
        Path("/kaggle/input/xlsr-aasist-antispoofing"),
    ]
    names = ["Best_LA_model_for_DF.pth", "Best_LA_model_for_LA.pth", "Best_LA_model_for_ITW.pth"]
    for base in bases:
        for name in names:
            if (base / name).exists():
                return base / name
        if base.exists():
            for path in base.glob("*.pth"):
                return path
    raise FileNotFoundError(
        "Wav2Vec2-XLSR+AASIST pretrained weights not found. "
        "Upload TakHemlata 'Best_LA_model_for_DF.pth' as a Kaggle dataset under "
        "/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/xlsr_aasist/."
    )


def find_xlsr_nes2net_checkpoint() -> Path:
    """Locate Liu-Tianchi's pretrained wav2vec2+Nes2Net-X anti-spoofing weights."""
    bases = [
        Path("/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/wav2vec2_nes2net"),
        Path("/kaggle/input/sdd-survey/checkpoints/wav2vec2_nes2net"),
        Path("/kaggle/input/wav2vec2-nes2net"),
        Path("/kaggle/input/nes2net-asvspoof-itw"),
    ]
    for base in bases:
        if base.exists():
            for pattern in ("*avg_ckpt*.pth", "*.pth", "*.pt"):
                for path in sorted(base.glob(pattern)):
                    return path
    raise FileNotFoundError(
        "wav2vec2+Nes2Net pretrained weights not found. "
        "Upload Liu-Tianchi's averaged checkpoint as a Kaggle dataset under "
        "/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/wav2vec2_nes2net/."
    )


def _strip_state_prefix(state: dict) -> dict:
    out = {}
    for k, v in state.items():
        if k.startswith("module."):
            out[k[len("module."):]] = v
        else:
            out[k] = v
    return out


def _reset_module_namespace(prefixes: list[str]) -> None:
    for name in list(sys.modules):
        if name in prefixes or any(name.startswith(p + ".") for p in prefixes):
            del sys.modules[name]


def _load_required_state(model: nn.Module, state: dict, model_name: str) -> None:
    """Load a checkpoint and fail on any architecture mismatch.

    Survey EER values are only useful when the full pretrained checkpoint is
    loaded. Silent partial loading can look successful while leaving random
    layers in the model, so missing/unexpected keys are fatal.
    """
    missing, unexpected = model.load_state_dict(_strip_state_prefix(state), strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f"{model_name} checkpoint does not match the model architecture. "
            f"missing={list(missing)[:10]} unexpected={list(unexpected)[:10]}"
        )


def load_xlsr_aasist_model() -> nn.Module:
    """Load Wav2Vec2-XLSR (300M) + AASIST from TakHemlata/SSL_Anti-spoofing."""
    repo_dir = ensure_ssl_aasist_repo()
    ensure_repo_fairseq(repo_dir)
    _link_xlsr_into(repo_dir)
    cwd = Path.cwd()
    try:
        os.chdir(repo_dir)
        _reset_module_namespace(["model", "models"])
        from model import Model as XLSRAASIST  # type: ignore
        import argparse
        model = XLSRAASIST(argparse.Namespace(), DEVICE).to(DEVICE)
        ckpt = find_xlsr_aasist_checkpoint()
        print(f"loading Wav2Vec2-XLSR+AASIST weights: {ckpt}")
        state = torch.load(ckpt, map_location=DEVICE)
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        _load_required_state(model, state, "XLS-R+AASIST")
        return model.eval()
    finally:
        os.chdir(cwd)


def load_xlsr_nes2net_model() -> nn.Module:
    """Load wav2vec2 (XLS-R 300M) + Nes2Net-X from Liu-Tianchi/Nes2Net_ASVspoof_ITW."""
    repo_dir = ensure_nes2net_repo()
    ensure_repo_fairseq(repo_dir)
    _link_xlsr_into(repo_dir)
    cwd = Path.cwd()
    try:
        os.chdir(repo_dir)
        _reset_module_namespace(["model_scripts"])
        from model_scripts.wav2vec2_Nes2Net_X import wav2vec2_Nes2Net_no_Res_w_allT  # type: ignore
        import argparse
        # Defaults from the repo training command:
        # --pool_func mean --SE_ratio 1 --Nes_ratio 8 8
        args = argparse.Namespace(
            n_output_logits=2,
            Nes_ratio=[8, 8],
            dilation=2,
            pool_func="mean",
            SE_ratio=1,
        )
        model = wav2vec2_Nes2Net_no_Res_w_allT(args, DEVICE).to(DEVICE)
        ckpt = find_xlsr_nes2net_checkpoint()
        print(f"loading wav2vec2+Nes2Net-X weights: {ckpt}")
        state = torch.load(ckpt, map_location=DEVICE)
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        elif isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        _load_required_state(model, state, "XLS-R+Nes2Net")
        return model.eval()
    finally:
        os.chdir(cwd)


@torch.no_grad()
def predict_ssl_pair_batch(model, waves: torch.Tensor) -> np.ndarray:
    """SSL models return 2-class logits ordered [spoof, bonafide]."""
    logits = model(waves.to(DEVICE))
    if not torch.is_tensor(logits) or logits.dim() != 2 or logits.shape[1] != 2:
        raise RuntimeError(f"Unexpected SSL model output shape/type: {type(logits)} {getattr(logits, 'shape', None)}")
    return torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()


def find_lfcc_checkpoint() -> Path:
    candidates = [
        Path("/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/lfcc_lcnn/lfcc_lcnn.pth"),
        Path("/kaggle/input/sdd-survey/checkpoints/lfcc_lcnn/lfcc_lcnn.pth"),
        Path("/kaggle/working/checkpoints/lfcc_lcnn/lfcc_lcnn.pth"),
        Path("results/checkpoints/lfcc_lcnn/lfcc_lcnn.pth"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("LFCC+LCNN checkpoint was not found. Attach sdd-survey or copy lfcc_lcnn.pth.")


def load_lfcc_lcnn_model() -> nn.Module:
    model = SimpleLCNN().to(DEVICE)
    ckpt = find_lfcc_checkpoint()
    print(f"loading LFCC+LCNN checkpoint: {ckpt}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model.eval()


def load_audio_16k(audio_ref):
    """Load path, bytes, or HF audio dict into mono 16 kHz waveform with shape (1, T)."""
    if isinstance(audio_ref, (str, Path)):
        wav, sr = torchaudio.load(str(audio_ref))
    elif isinstance(audio_ref, bytes):
        array, sr = sf.read(io.BytesIO(audio_ref), dtype="float32")
        if array.ndim == 1:
            wav = torch.from_numpy(array).float().unsqueeze(0)
        else:
            wav = torch.from_numpy(array).float().T
    elif isinstance(audio_ref, dict) and "array" in audio_ref and "sampling_rate" in audio_ref:
        wav = torch.from_numpy(audio_ref["array"]).float().unsqueeze(0)
        sr = int(audio_ref["sampling_rate"])
    else:
        raise TypeError(f"unsupported audio ref: {type(audio_ref)}")

    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
        wav = torchaudio.transforms.Resample(sr, 16000)(wav)
    return wav


def prepare_waveform(wav: torch.Tensor, cut: int = 64600) -> torch.Tensor:
    if wav.shape[1] < cut:
        wav = F.pad(wav, (0, cut - wav.shape[1]))
    else:
        wav = wav[:, :cut]
    return wav.squeeze(0)


def prepare_lfcc(wav: torch.Tensor, max_frames: int = 400) -> torch.Tensor:
    feat = LFCC_TRANSFORM(wav)
    if feat.shape[2] < max_frames:
        feat = F.pad(feat, (0, max_frames - feat.shape[2]))
    else:
        feat = feat[:, :, :max_frames]
    return feat.squeeze(0)


class WaveformDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        try:
            wav = prepare_waveform(load_audio_16k(row["audio_path"]))
            return wav, int(row["label"]), str(row["utt_id"]), 0
        except Exception:
            return torch.zeros(64600), int(row["label"]), str(row["utt_id"]), 1


class LFCCDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        try:
            feat = prepare_lfcc(load_audio_16k(row["audio_path"]))
            return feat, int(row["label"]), str(row["utt_id"]), 0
        except Exception:
            return torch.zeros(60, 400), int(row["label"]), str(row["utt_id"]), 1


@torch.no_grad()
def predict_aasist_batch(model, waves: torch.Tensor) -> np.ndarray:
    """AASIST and AASIST-L output logits [spoof, bonafide]."""
    _, logits = model(waves.to(DEVICE))
    return logits.softmax(dim=1)[:, 1].detach().cpu().numpy()


@torch.no_grad()
def predict_aasist3_batch(model, waves: torch.Tensor) -> np.ndarray:
    """AASIST3 output logits [spoof, bonafide]. Use index 1 as bonafide score."""
    logits = model(waves.to(DEVICE))
    return torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()


@torch.no_grad()
def predict_lfcc_batch(model, feats: torch.Tensor) -> np.ndarray:
    if feats.dim() == 3:
        feats = feats.unsqueeze(1)
    logits = model(feats.to(DEVICE))
    return torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()


MODEL_REGISTRY = {
    "AASIST": {
        "loader": lambda: load_aasist_model("AASIST.conf", "AASIST.pth"),
        "dataset": WaveformDataset,
        "predict": predict_aasist_batch,
        "batch_size": 64,
    },
    "AASIST-L": {
        "loader": lambda: load_aasist_model("AASIST-L.conf", "AASIST-L.pth"),
        "dataset": WaveformDataset,
        "predict": predict_aasist_batch,
        "batch_size": 64,
    },
    "AASIST3": {
        "loader": load_aasist3_model,
        "dataset": WaveformDataset,
        "predict": predict_aasist3_batch,
        "batch_size": 64,
    },
    "LFCC+LCNN": {
        "loader": load_lfcc_lcnn_model,
        "dataset": LFCCDataset,
        "predict": predict_lfcc_batch,
        "batch_size": 128,
    },
    "XLS-R+AASIST": {
        "loader": load_xlsr_aasist_model,
        "dataset": WaveformDataset,
        "predict": predict_ssl_pair_batch,
        "batch_size": 8,
    },
    "XLS-R+Nes2Net": {
        "loader": load_xlsr_nes2net_model,
        "dataset": WaveformDataset,
        "predict": predict_ssl_pair_batch,
        "batch_size": 8,
    },
}


COMPOUND_GROUPS = [
    ("WaveformDataset", ["XLS-R+AASIST", "XLS-R+Nes2Net"]),
]

In [ ]:
def evaluate_model_on_dataframe(
    model_name: str,
    df: pd.DataFrame,
    results: dict,
    output_pkl: Path,
    output_dir: Path,
    force_eval: bool = False,
    partial_save_every: int = 5000,
    num_workers: int = 2,
    partial_input_dirs: list[Path] | None = None,
) -> dict:
    """Evaluate one model on local files with partial resume and unified pkl writes."""
    if model_name in results and not force_eval:
        print(f"{model_name}: already present in results.pkl, skipping")
        return results[model_name]

    entry = MODEL_REGISTRY[model_name]
    partial = load_partial(output_dir, model_name, partial_input_dirs)
    scores, labels, utt_ids = [], [], []
    completed = set()
    if partial:
        scores = partial["scores"]
        labels = partial["labels"]
        utt_ids = partial["utt_ids"]
        completed = set(utt_ids)
        print(f"{model_name}: resume partial with {len(completed):,} completed utterances")

    todo_df = df[~df["utt_id"].astype(str).isin(completed)].reset_index(drop=True)
    print(f"{model_name}: todo={len(todo_df):,} already_done={len(completed):,}")

    model = entry["loader"]()
    dataset = entry["dataset"](todo_df)
    loader = DataLoader(
        dataset,
        batch_size=entry["batch_size"],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    errors = 0
    last_save = len(utt_ids)
    t0 = time.time()
    try:
        for batch_inputs, batch_labels, batch_utt_ids, batch_is_error in tqdm(loader, desc=model_name):
            batch_scores = entry["predict"](model, batch_inputs)
            ok_mask = batch_is_error.numpy() == 0
            errors += int((~ok_mask).sum())
            for i, ok in enumerate(ok_mask):
                if not ok:
                    continue
                scores.append(float(batch_scores[i]))
                labels.append(int(batch_labels[i].item()))
                utt_ids.append(str(batch_utt_ids[i]))

            if len(utt_ids) - last_save >= partial_save_every:
                save_partial(output_dir, model_name, scores, labels, utt_ids)
                last_save = len(utt_ids)
    except KeyboardInterrupt:
        save_partial(output_dir, model_name, scores, labels, utt_ids)
        print(f"{model_name}: interrupted; partial saved")
        raise
    finally:
        release_model(model)
        model = None

    eer = compute_eer(scores, labels)
    result = {
        "eer": eer,
        "scores": np.asarray(scores, dtype=np.float64),
        "labels": np.asarray(labels, dtype=np.int64),
    }
    results[model_name] = result
    save_results_pickle(results, output_pkl)
    clear_partial(output_dir, model_name)
    print(f"{model_name}: EER={eer:.4f}% N={len(scores):,} errors={errors:,} elapsed={(time.time()-t0)/60:.1f} min")
    return result


def evaluate_models_compound(
    model_names: list[str],
    df: pd.DataFrame,
    results: dict,
    output_pkl: Path,
    output_dir: Path,
    force_eval: dict[str, bool] | None = None,
    partial_save_every: int = 5000,
    num_workers: int = 2,
    partial_input_dirs: list[Path] | None = None,
) -> None:
    """Run multiple models in one pass over the same data loader.

    All listed models must share the same dataset class. Each model has its own
    partial.npz checkpoint and is added to results.pkl independently when finished.
    The compound runner saves audio I/O versus running models sequentially because
    each utterance is decoded once and each model in the group consumes the same
    batch tensor. The XLS-R front-ends inside SSL models are NOT shared because
    each pretrained checkpoint contains independently fine-tuned SSL weights.
    """
    force_eval = force_eval or {}
    pending = []
    for name in model_names:
        if name in results and not force_eval.get(name, False):
            print(f"{name}: already present in results.pkl, skipping")
        else:
            pending.append(name)
    if not pending:
        return

    dataset_classes = {MODEL_REGISTRY[n]["dataset"] for n in pending}
    if len(dataset_classes) > 1:
        raise ValueError(f"compound run requires identical dataset class, got {dataset_classes}")
    DatasetCls = dataset_classes.pop()

    partials = {}
    completed = {}
    for name in pending:
        partial = load_partial(output_dir, name, partial_input_dirs)
        if partial:
            partials[name] = partial
            completed[name] = set(partial["utt_ids"])
            print(f"{name}: resume partial with {len(partial['utt_ids']):,} completed")
        else:
            partials[name] = {"scores": [], "labels": [], "utt_ids": []}
            completed[name] = set()

    todo_mask = df["utt_id"].astype(str).map(
        lambda u: any(u not in completed[n] for n in pending)
    )
    todo_df = df[todo_mask].reset_index(drop=True)
    print(f"compound[{'+'.join(pending)}]: todo={len(todo_df):,}")

    models = {}
    for name in pending:
        print(f"loading {name}")
        models[name] = MODEL_REGISTRY[name]["loader"]()
    batch_size = min(MODEL_REGISTRY[n]["batch_size"] for n in pending)

    dataset = DatasetCls(todo_df)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    last_save = {n: len(partials[n]["utt_ids"]) for n in pending}
    errors = 0
    t0 = time.time()
    desc = "+".join(pending)
    try:
        for batch_inputs, batch_labels, batch_utt_ids, batch_is_error in tqdm(loader, desc=desc):
            ok_mask = batch_is_error.numpy() == 0
            errors += int((~ok_mask).sum())
            for name in pending:
                entry = MODEL_REGISTRY[name]
                batch_scores = entry["predict"](models[name], batch_inputs)
                for i, ok in enumerate(ok_mask):
                    if not ok:
                        continue
                    utt_id = str(batch_utt_ids[i])
                    if utt_id in completed[name]:
                        continue
                    partials[name]["scores"].append(float(batch_scores[i]))
                    partials[name]["labels"].append(int(batch_labels[i].item()))
                    partials[name]["utt_ids"].append(utt_id)
                    completed[name].add(utt_id)
                if len(partials[name]["utt_ids"]) - last_save[name] >= partial_save_every:
                    save_partial(output_dir, name, partials[name]["scores"], partials[name]["labels"], partials[name]["utt_ids"])
                    last_save[name] = len(partials[name]["utt_ids"])
    except KeyboardInterrupt:
        for name in pending:
            save_partial(output_dir, name, partials[name]["scores"], partials[name]["labels"], partials[name]["utt_ids"])
        print("compound: interrupted; partials saved")
        raise
    finally:
        for name in pending:
            release_model(models[name])
            models[name] = None

    for name in pending:
        eer = compute_eer(partials[name]["scores"], partials[name]["labels"])
        results[name] = {
            "eer": eer,
            "scores": np.asarray(partials[name]["scores"], dtype=np.float64),
            "labels": np.asarray(partials[name]["labels"], dtype=np.int64),
        }
        save_results_pickle(results, output_pkl)
        clear_partial(output_dir, name)
        print(f"{name}: EER={eer:.4f}% N={len(partials[name]['scores']):,}")
    print(f"compound[{desc}] elapsed={(time.time()-t0)/60:.1f} min, errors={errors:,}")


def run_eval_plan(
    enabled_models: list[str],
    df: pd.DataFrame,
    results: dict,
    output_pkl: Path,
    output_dir: Path,
    force_eval: dict[str, bool] | None = None,
    partial_save_every: int = 5000,
    num_workers: int = 2,
    compound_enabled: bool = False,
    partial_input_dirs: list[Path] | None = None,
) -> None:
    """Schedule single-model and compound runs for the configured ENABLED_MODELS.

    Compound mode is opt-in because loading two XLS-R models at once can exceed Kaggle
    T4/P100 memory. When disabled, every model runs sequentially with the same partial
    resume behavior. When enabled, models declared in COMPOUND_GROUPS share one loader.
    """
    force_eval = force_eval or {}
    grouped = set()
    if compound_enabled:
        for _tag, group in COMPOUND_GROUPS:
            members = [m for m in group if m in enabled_models]
            if len(members) >= 2:
                evaluate_models_compound(
                    model_names=members,
                    df=df,
                    results=results,
                    output_pkl=output_pkl,
                    output_dir=output_dir,
                    force_eval=force_eval,
                    partial_save_every=partial_save_every,
                    num_workers=num_workers,
                    partial_input_dirs=partial_input_dirs,
                )
                grouped.update(members)
    for name in enabled_models:
        if name in grouped:
            continue
        evaluate_model_on_dataframe(
            model_name=name,
            df=df,
            results=results,
            output_pkl=output_pkl,
            output_dir=output_dir,
            force_eval=force_eval.get(name, False),
            partial_save_every=partial_save_every,
            num_workers=num_workers,
            partial_input_dirs=partial_input_dirs,
        )

In [ ]:
run_eval_plan(
    enabled_models=ENABLED_MODELS,
    df=eval_df,
    results=results,
    output_pkl=OUTPUT_PKL,
    output_dir=OUTPUT_DIR,
    force_eval=FORCE_EVAL,
    partial_save_every=PARTIAL_SAVE_EVERY,
    num_workers=NUM_WORKERS,
    compound_enabled=RUN_COMPOUND_SSL_MODELS,
    partial_input_dirs=PARTIAL_INPUT_DIRS,
)

save_results_pickle(results, OUTPUT_PKL)
print("final summary")
for name, result in results.items():
    print(f"{name:20s} EER={result['eer']:.4f}% N={len(result['scores']):,}")